# AI Video Cutter — Data Flow & Usage

## Data Flow

```
Video file(s)
    │
    ▼
[1] config/default.yaml  ──►  Settings (Pydantic)
                                  │ config_hash (SHA-256 of video section)
                                  │ used to decide whether reprocessing is needed
    │
    ▼
[2] ProjectStorage.create_project(name, [video_paths])
    │  • hashes each video (first 64 KB + file size → 16-char hex)
    │  • copies default.yaml → projects/{name}/config.yaml
    │  • writes videos/manifest.json  (all per-step processing status)
    │  • writes project.json
    ▼
    projects/{name}/
    ├── config.yaml
    ├── project.json
    ├── videos/
    │   ├── manifest.json            ← per-video, per-step timestamps + config_hash
    │   └── {video_hash}/
    │       ├── meta.json            ← ffprobe output
    │       ├── flow/metrics.json    ← per-frame optical flow metrics
    │       ├── segments/segments.json
    │       └── descriptions/vlm.json  ← VideoDescription (whole-video VLM)
    ├── analysis/
    │   ├── frame_metrics.json
    │   └── combined.json            ← merged SegmentBase + SegmentDescription
    ├── storyboard/
    │   ├── v1.json  ◄── latest.json (symlink)
    │   └── v2.json
    └── timeline/
        └── v1.json  ◄── latest.json
```

```
[3] Pipeline steps (each checks is_step_current before running):

    OpticalFlowStep     → ctx.raw_signal, ctx.frame_metrics
    PreprocessSignalStep → ctx.preprocessed_signal
    SegmentScenesStep   → ctx.segments  (list[SegmentBase])
    PersistStep         → saves to storage, marks steps complete in manifest
```

```
[4] VLM enrichment (future / separate step):

    SegmentBase  ──►  VLM prompt  ──►  SegmentDescription
    VideoFile    ──►  VLM prompt  ──►  VideoDescription

    build_combined_view(segments, descriptions) → list[Segment]
    (joined on segment_id, vlm field holds list of descriptions)
```

In [ ]:
import sys
from pathlib import Path

# Add src/ to path when running from notebooks/
sys.path.insert(0, str(Path("..") / "src"))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from core.config import Settings
from core.storage import ProjectStorage
from core.schemas.segment import SegmentBase, SegmentDescription, Segment, build_combined_view
from core.schemas.video_description import VideoDescription

REPO_ROOT = Path("..")
CONFIG_PATH = REPO_ROOT / "config" / "default.yaml"
STORAGE_ROOT = REPO_ROOT / "local" / "data" / "projects"

## 1. Configuration

`config/default.yaml` is the canonical source of defaults. On project creation it is copied into the project folder as `config.yaml` so each project is self-contained.

**What you can configure:**

| Section | Key parameters |
|---------|---------------|
| `video` | `target_fps`, `target_width`, `hwaccel`, `optical_flow.*`, `segmentation.*` |
| `vlm` | `provider`, `model`, `temperature` |
| `storyboard` | `model`, `review_threshold` |
| `editor` | `model`, `similarity_threshold` |

**`config_hash`** — SHA-256 of the `video` section only. Stored in `manifest.json` next to each processed step. If you change any video parameter (fps, width, flow params, segmentation penalties) and re-run, every affected step will be reprocessed automatically. Changing `vlm` or `storyboard` settings does **not** invalidate optical-flow results.

In [ ]:
# Load and inspect the default config
settings = Settings.load(CONFIG_PATH)

print("=== Video config ===")
print(f"  target_fps    : {settings.video.target_fps}")
print(f"  target_width  : {settings.video.target_width}")
print(f"  hwaccel       : {settings.video.hwaccel}")
print(f"  fd_penalty    : {settings.video.segmentation.fd_penalty}")
print(f"  subseg_penalty: {settings.video.segmentation.subseg_penalty}")
print(f"  savgol_window : {settings.video.segmentation.savgol_window}")
print()
print("=== VLM config ===")
print(f"  provider  : {settings.vlm.provider}")
print(f"  model     : {settings.vlm.model}")
print(f"  temperature: {settings.vlm.temperature}")
print()
print(f"config_hash (video section): {settings.config_hash}")
print()

# Demonstrate that config_hash only changes when VIDEO params change
changed = settings.model_copy(
    update={"video": settings.video.model_copy(update={"target_fps": 2.0})}
)
print(f"config_hash after fps=2.0  : {changed.config_hash}  ← DIFFERENT")

vlm_changed = settings.model_copy(
    update={"vlm": settings.vlm.model_copy(update={"temperature": 0.9})}
)
print(f"config_hash after temp=0.9 : {vlm_changed.config_hash}  ← SAME (vlm change ignored)")

## 2. Running the Pipeline

### CLI (simplest)

```bash
# Process a video with defaults from config/default.yaml
vc process path/to/video.MP4

# Custom project name and storage location
vc process path/to/video.MP4 --project-name my_project --storage-root local/data/projects

# Override specific video params (all others come from config/default.yaml)
vc process path/to/video.MP4 --fd-penalty 5.0 --target-width 1280

# Use a different config file entirely
vc process path/to/video.MP4 --config config/high_res.yaml

# Hardware acceleration
vc process path/to/video.MP4 --hwaccel videotoolbox   # macOS
vc process path/to/video.MP4 --hwaccel cuda           # NVIDIA
```

### Python API (for notebooks / scripting)

```python
from core.config import Settings
from core.storage import ProjectStorage
from core.schemas.video import ProcessingConfig, SegmentationConfig
from video.pipeline import default_pipeline

settings = Settings.load("config/default.yaml")
storage  = ProjectStorage(root="local/data/projects", default_config="config/default.yaml")

proc = ProcessingConfig(
    target_fps   = settings.video.target_fps,
    target_width = settings.video.target_width,
    hwaccel      = settings.video.hwaccel,
)
seg = SegmentationConfig(
    fd_penalty    = settings.video.segmentation.fd_penalty,
    subseg_penalty= settings.video.segmentation.subseg_penalty,
    savgol_window = settings.video.segmentation.savgol_window,
    savgol_poly   = settings.video.segmentation.savgol_poly,
)

ctx = default_pipeline(proc, seg, storage, config=settings).run(
    "path/to/video.MP4", project_name="my_project"
)
print(f"Project: {ctx.project_id}   Segments: {len(ctx.segments)}")
```

## 3. Exploring Processed Projects

Set `PROJECT_NAME` to the name of a project you've already processed with `vc process`.

In [ ]:
storage = ProjectStorage(root=STORAGE_ROOT, default_config=CONFIG_PATH)

projects = storage.list_projects()
if projects:
    print(f"Found {len(projects)} project(s):")
    for p in projects:
        print(f"  • {p.name}  status={p.status.value}  videos={len(p.video_files)}")
else:
    print("No projects found. Run: vc process path/to/video.MP4 --storage-root local/data/projects")

# ── Set your project name here ────────────────────────────────────────────────
PROJECT_NAME = projects[0].name if projects else None
print(f"\nUsing project: {PROJECT_NAME}")

In [ ]:
if PROJECT_NAME is None:
    raise RuntimeError("No project found — process a video first (see cell above).")

# ── Manifest: per-video processing status ─────────────────────────────────────
manifest_path = storage.get_project_path(PROJECT_NAME) / "videos" / "manifest.json"
manifest = json.loads(manifest_path.read_text())

print(f"Project: {PROJECT_NAME}")
print(f"Videos in manifest: {len(manifest['videos'])}\n")

for vh, entry in manifest["videos"].items():
    print(f"  {vh}  ({entry['filename']})")
    print(f"    config_hash : {entry['config_hash']}")
    for step, ts in entry["processing"].items():
        status = "✓" if ts else "·"
        print(f"    {status} {step:<15} {ts or '—'}")
    print()

VIDEO_HASH = next(iter(manifest["videos"]))

## 4. Segments

Each `SegmentBase` has:
- **`segment_id`** — 8-char deterministic hash: `sha256(f"{source_video}:{index}")[:8]`
- **`video_file`** — original filename
- **`source_video`** — 16-char video content-hash
- **`start` / `end`** — timestamps in seconds
- **`camera_movements`** — list of `CameraMovement` (one per sub-segment detected by Ruptures l1)
  - per-channel (pan/tilt/zoom): entry/exit velocity, monotonicity, mean absolute derivative, std derivative

In [ ]:
# Load segments from content-addressed path
segments_path = f"videos/{VIDEO_HASH}/segments/segments.json"
segments: list[SegmentBase] = storage.load_json(PROJECT_NAME, segments_path, SegmentBase)

print(f"Loaded {len(segments)} segments for video {VIDEO_HASH}\n")

# ── Segment overview table ────────────────────────────────────────────────────
seg_rows = []
for s in segments:
    seg_rows.append({
        "segment_id"    : s.segment_id,
        "start"         : round(s.start, 2),
        "end"           : round(s.end, 2),
        "duration"      : round(s.end - s.start, 2),
        "n_movements"   : len(s.camera_movements),
    })

df_segs = pd.DataFrame(seg_rows)
display(df_segs)

In [ ]:
# ── Camera movements flat table ───────────────────────────────────────────────
move_rows = []
for s in segments:
    for m in s.camera_movements:
        move_rows.append({
            "segment_id"       : s.segment_id,
            "movement_id"      : m.movement_id,
            "start_time"       : round(m.start_time, 2),
            "end_time"         : round(m.end_time, 2),
            "pan_entry_vel"    : round(m.pan_entry_vel, 4),
            "pan_exit_vel"     : round(m.pan_exit_vel, 4),
            "tilt_entry_vel"   : round(m.tilt_entry_vel, 4),
            "tilt_exit_vel"    : round(m.tilt_exit_vel, 4),
            "zoom_entry_vel"   : round(m.zoom_entry_vel, 4),
            "zoom_exit_vel"    : round(m.zoom_exit_vel, 4),
            "pan_monotonicity" : round(m.pan_monotonicity, 3),
            "tilt_monotonicity": round(m.tilt_monotonicity, 3),
        })

df_moves = pd.DataFrame(move_rows)
display(df_moves)

In [ ]:
# ── Signal plot with scene and movement boundaries ────────────────────────────
fm_raw = storage.load_json(PROJECT_NAME, "analysis/frame_metrics.json")
df_fm = pd.DataFrame([
    {
        "timestamp"      : r["timestamp"],
        "pan"            : r["metrics"].get("pan"),
        "tilt"           : r["metrics"].get("tilt"),
        "zoom"           : r["metrics"].get("zoom"),
        "flow_magnitude" : r["metrics"].get("flow_magnitude"),
    }
    for r in fm_raw
]).dropna(subset=["pan", "tilt", "zoom"])

fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
channels = ["pan", "tilt", "zoom", "flow_magnitude"]
colors   = ["steelblue", "darkorange", "forestgreen", "mediumpurple"]

for ax, ch, col in zip(axes, channels, colors):
    ax.plot(df_fm["timestamp"], df_fm[ch], color=col, linewidth=0.8, label=ch)
    ax.set_ylabel(ch)

    for s in segments:
        ax.axvline(s.start, color="red", linewidth=1.5, alpha=0.7)
        for m in s.camera_movements:
            ax.axvline(m.start_time, color="grey", linewidth=0.7, linestyle="--", alpha=0.5)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Camera motion signal — red: scene boundary · grey: movement boundary")
scene_p = mpatches.Patch(color="red",  label="Scene boundary")
move_p  = mpatches.Patch(color="grey", label="Movement boundary")
fig.legend(handles=[scene_p, move_p], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Entry vs exit velocity scatter per channel ────────────────────────────────
if len(df_moves) > 0:
    seg_id_to_int = {s: i for i, s in enumerate(df_moves["segment_id"].unique())}
    df_moves["seg_color"] = df_moves["segment_id"].map(seg_id_to_int)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, ch in zip(axes, ["pan", "tilt", "zoom"]):
        sc = ax.scatter(
            df_moves[f"{ch}_entry_vel"],
            df_moves[f"{ch}_exit_vel"],
            c=df_moves["seg_color"], cmap="tab10", alpha=0.8, s=60,
        )
        ax.axline((0, 0), slope=1, color="grey", linewidth=0.8, linestyle="--")
        ax.axhline(0, color="black", linewidth=0.4)
        ax.axvline(0, color="black", linewidth=0.4)
        ax.set_xlabel(f"{ch} entry_vel")
        ax.set_ylabel(f"{ch} exit_vel")
        ax.set_title(f"{ch} — entry vs exit velocity")
        plt.colorbar(sc, ax=ax, label="segment (int index)")
    fig.suptitle("Camera movement velocity profile (each dot = one movement sub-segment)")
    plt.tight_layout()
    plt.show()
else:
    print("No camera movements to plot.")

## 5. VLM Descriptions & Merged View

### Schema

`SegmentDescription` mirrors the VLM output format:

```python
SegmentDescription(
    segment_id    = "07b87d32",   # echoed from SegmentBase.segment_id
    description   = "High angle wide shot of SUV on dirt road...",
    technical_specs = TechnicalSpecs(
        framing  = "Wide Shot",
        movement = "Tracking Shot",
        angle    = "High Angle",
        reasoning = TechnicalSpecsReasoning(framing="...", movement="...", angle="..."),
    ),
    color_profile = ColorProfile(
        dominant_colors = ["#6B7B5A", "#9A8C82", "#EAEAEA"],
        lighting_type   = "Overcast",
        temperature     = "cool",
    ),
    highlights = [Highlight(
        description = "SUV crosses stream",
        keywords    = ["stream", "crossing", "off-road"],
        start       = "00:00:08.000",
        end         = "00:00:12.750",
    )],
    quality_score = QualityScore(rating="excellent", reasoning="Smooth stable drone footage."),
    segment_tags  = ["aerial", "drone", "mountains", "off-road"],
)
```

`VideoDescription` covers the whole video:

```python
VideoDescription(
    video_id  = "abc123ef",          # video content-hash
    video_file = "DJI_0135.MP4",
    vlm = VideoVlm(
        description   = "Drone footage tracking SUV through mountainous terrain...",
        key_subjects  = [["SUV", "Dark grey vehicle on a dirt road."]],
        tone          = ["cinematic", "adventurous", "serene"],
        genre_or_type = "aerial_videography",
        tags          = ["drone", "suv", "mountains", "off-road"],
    ),
)
```

Both are stored as Pydantic models and round-trip through `storage.save_json` / `storage.load_json`.

### Merged view

`build_combined_view(segments, descriptions) → list[Segment]`

- Joins on `segment_id`
- `Segment.vlm` is a **list** — supports multiple VLM runs / revisions per segment
- Segments with no description get `vlm=[]`

In [ ]:
from core.schemas.segment import TechnicalSpecs, TechnicalSpecsReasoning, ColorProfile, Highlight, QualityScore

# ── Simulate a VLM response for the first segment ────────────────────────────
if segments:
    first_seg = segments[0]
    fake_description = SegmentDescription(
        segment_id = first_seg.segment_id,
        description = (
            f"Wide shot of scene starting at {first_seg.start:.1f}s. "
            f"Camera performs a {first_seg.camera_movements[0].pan_monotonicity:.2f}-monotonicity pan."
            if first_seg.camera_movements else
            f"Wide establishing shot from {first_seg.start:.1f}s to {first_seg.end:.1f}s."
        ),
        technical_specs = TechnicalSpecs(
            framing  = "Wide Shot",
            movement = "Tracking Shot",
            angle    = "High Angle",
            reasoning = TechnicalSpecsReasoning(
                framing  = "Captures the full environment and scale.",
                movement = "Camera follows subject motion.",
                angle    = "Elevated perspective shows terrain context.",
            ),
        ),
        color_profile = ColorProfile(
            dominant_colors = ["#6B7B5A", "#9A8C82"],
            lighting_type   = "Overcast",
            temperature     = "cool",
        ),
        highlights = [Highlight(
            description = "Key moment in scene",
            keywords    = ["motion", "landscape", "aerial"],
            start       = f"00:00:{int(first_seg.start):02d}.000",
            end         = f"00:00:{int((first_seg.start + first_seg.end) / 2):02d}.000",
        )],
        quality_score = QualityScore(rating="good", reasoning="Smooth aerial footage."),
        segment_tags  = ["aerial", "drone", "landscape"],
    )

    # ── Merge: build_combined_view joins on segment_id ────────────────────────
    all_descriptions = [fake_description]   # in practice: loaded from storage
    merged: list[Segment] = build_combined_view(segments, all_descriptions)

    print("Merged segments:")
    for m in merged:
        vlm_count = len(m.vlm)
        desc_preview = m.vlm[0].description[:80] + "…" if m.vlm else "(no VLM yet)"
        print(f"  {m.segment_id}  {m.start:.1f}s–{m.end:.1f}s  "
              f"movements={len(m.camera_movements)}  vlm_runs={vlm_count}")
        print(f"    {desc_preview}")

    print()
    print("Serialised first merged segment (truncated):")
    d = merged[0].model_dump(mode="json")
    print(json.dumps({k: v for k, v in d.items() if k != "camera_movements"}, indent=2))
else:
    print("No segments loaded — process a video first.")

In [ ]:
# ── Saving descriptions back to storage ──────────────────────────────────────
#
# Once you have real VLM output, persist it:
#
#   storage.save_json(
#       PROJECT_NAME,
#       f"videos/{VIDEO_HASH}/descriptions/vlm.json",
#       [desc],   # list of SegmentDescription
#   )
#
# Load it back:
#
#   descriptions = storage.load_json(
#       PROJECT_NAME,
#       f"videos/{VIDEO_HASH}/descriptions/vlm.json",
#       SegmentDescription,
#   )
#   merged = build_combined_view(segments, descriptions)
#
# Save combined view for downstream consumers:
#
#   storage.save_json(PROJECT_NAME, "analysis/combined.json", merged)
#
# Save a versioned storyboard:
#
#   from core.schemas.storyboard import Storyboard, StoryboardScene
#   sb = Storyboard(scenes=[StoryboardScene(index=i, segment_index=i) for i in range(len(merged))])
#   version = storage.save_versioned(PROJECT_NAME, "storyboard", sb)
#   print(f"Saved storyboard v{version}")
#   # projects/{name}/storyboard/v{n}.json  ←  latest.json symlink

print("Storage API reference printed above — uncomment to run.")